In [11]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import xarray as xr
from jiflr import ROOT
from jiflr.data import select_sensors, merge_sensor_datasets
from itertools import combinations

In [26]:
# Load pendant data
ds_pendants = xr.open_dataset(ROOT / "data/2025/processed/lvl1/lvl1_on_ice.nc")
ds_pendants = select_sensors(ds_pendants, height="2m", shielding="shielded")
# Drop sensors missing coordinates (e.g. G03A, G03B)
ds_pendants = ds_pendants.isel(sensor_idx=~np.isnan(ds_pendants.latitude.values))

# Load intensive site data and filter to shielded pendants
ds_intensive = xr.open_dataset(
    ROOT / "data/2025/processed/lvl1/lvl1_on_ice_intensive.nc"
)
ds_intensive = select_sensors(ds_intensive, height="2m", shielding="shielded")

# Merge datasets
ds = merge_sensor_datasets(ds_pendants, ds_intensive)

# Build site metadata from sensor coordinates
site_ids = ds.site_id.values
site_meta = {
    str(sid): {
        "lat": float(ds.latitude.isel(sensor_idx=i).values),
        "lon": float(ds.longitude.isel(sensor_idx=i).values),
        "elev": int(ds.elevation.isel(sensor_idx=i).values),
    }
    for i, sid in enumerate(site_ids)
}

da = ds.temp_c
da = da.dropna(dim="datetime", how="all")

# Filter to approx daytime hours (8am - 7pm)
hours = da.datetime.dt.hour
da = da.sel(datetime=(hours >= 8) & (hours < 19))

# Remove the mean temperature at every time period
da_mean = da.mean(dim="sensor_idx")
da = ds.temp_c - da_mean


In [27]:
def optimal_lag_between_sensors(da: xr.DataArray, max_lag: int = None) -> pd.DataFrame:
    """
    Compute the optimal cross-correlation lag between every pair of sensors.

    Parameters
    ----------
    da : xr.DataArray
        Dims: (datetime, sensor_idx). Should be anomaly (mean-removed) data.
    max_lag : int, optional
        Maximum lag (in time steps) to consider. Defaults to 10% of time series length.

    Returns
    -------
    pd.DataFrame
        Columns: sensor_a, sensor_b, lag, correlation
        lag > 0 means sensor_b leads sensor_a
        lag < 0 means sensor_a leads sensor_b
    """
    sensor_ids = da.sensor_idx.values
    site_ids = da.site_id.values
    times = da.datetime.values
    n_time = len(times)

    if max_lag is None:
        max_lag = n_time // 10

    # Convert to numpy, shape (n_time, n_sensors)
    arr = da.values  # (datetime, sensor_idx)

    records = []

    for i, j in combinations(range(len(sensor_ids)), 2):
        x = arr[:, i]
        y = arr[:, j]

        # Mask to times where both are valid
        valid = ~np.isnan(x) & ~np.isnan(y)
        x_v = x[valid]
        y_v = y[valid]

        if len(x_v) < 2 * max_lag + 1:
            records.append(
                {
                    "sensor_a": site_ids[i],
                    "sensor_b": site_ids[j],
                    "lag": np.nan,
                    "correlation": np.nan,
                    "corr_at_zero": np.nan,
                    "corr_diff": np.nan,
                }
            )
            continue

        # Normalize
        x_norm = (x_v - np.nanmean(x_v)) / (np.nanstd(x_v) + 1e-12)
        y_norm = (y_v - np.nanmean(y_v)) / (np.nanstd(y_v) + 1e-12)

        # Full cross-correlation via numpy
        # np.correlate gives xcorr[k] = sum_t x[t] * y[t + k]
        # mode='full' gives lags from -(n-1) to +(n-1)
        xcorr = np.correlate(x_norm, y_norm, mode="full") / len(x_norm)
        lags = np.arange(-(len(x_norm) - 1), len(x_norm))

        # Restrict to max_lag window
        mask = np.abs(lags) <= max_lag
        xcorr_windowed = xcorr[mask]
        lags_windowed = lags[mask]

        best_idx = np.argmax(np.abs(xcorr_windowed))
        best_lag = lags_windowed[best_idx]
        best_corr = xcorr_windowed[best_idx]

        zero_idx = np.where(lags_windowed == 0)[0][0]
        zero_corr = float(xcorr_windowed[zero_idx])

        records.append(
            {
                "sensor_a": site_ids[i],
                "sensor_b": site_ids[j],
                "lag": int(best_lag),
                "correlation": round(best_corr, 4),
                "corr_at_zero": round(float(zero_corr), 4),
                "corr_diff": round(best_corr - zero_corr, 4),
            }
        )

    df = pd.DataFrame(records)
    return df


# --- Run it ---
df_lags = optimal_lag_between_sensors(da, max_lag=96)  # e.g. 96 x 15min = 24h

# Convert lag from time steps to actual duration
dt = pd.to_timedelta(da.datetime.diff("datetime").median().values)
df_lags["lag_duration"] = df_lags["lag"].apply(
    lambda x: x * dt if pd.notna(x) else pd.NaT
)


In [28]:
df_lags[df_lags.correlation >= 0.5].sort_values("correlation", ascending=False)

,sensor_a,sensor_b,lag,correlation,corr_at_zero,corr_diff,lag_duration
318,A02,G02,1,0.8477,0.8433,0.0044,0 days 00:05:00
18,G01,G02,0,0.7852,0.7852,0.0000,0 days 00:00:00
118,F05,F06,1,0.7547,0.7544,0.0004,0 days 00:05:00
73,E03,E04,0,0.7249,0.7249,0.0000,0 days 00:00:00
198,A04,A03,0,0.6782,0.6782,0.0000,0 days 00:00:00
150,A05,Windward2,-1,0.6539,0.6537,0.0002,-1 days +23:55:00
69,E03,F06,-7,0.6472,0.6135,0.0337,-1 days +23:25:00
14,G01,A02,-2,0.6349,0.6332,0.0017,-1 days +23:50:00
239,C02,A02,-3,0.6275,0.6216,0.0058,-1 days +23:45:00
132,A05,A04,1,0.6236,0.6229,0.0008,0 days 00:05:00
